In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import glob
import os

# Use seaborn color palette for plotly
seaborn_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
                  '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

# 1️⃣ ระบุโฟลเดอร์หลัก
main_folder = "data_0/71_sensing_data"  # เปลี่ยนเป็น path ของคุณ

# 2️⃣ เดินเข้าไปทุกโฟลเดอร์ย่อยและอ่านไฟล์ CSV
df_list = []

# glob แบบ recursive
all_csv_files = glob.glob(os.path.join(main_folder, "**", "*.csv"), recursive=True)

for file in all_csv_files:
    temp_df = pd.read_csv(file)
    
    # ใช้ชื่อโฟลเดอร์ย่อยเป็น SerialNo ถ้า SerialNo ไม่มีในไฟล์
    if 'SerialNo' not in temp_df.columns:
        serial_no = os.path.basename(os.path.dirname(file))
        temp_df['SerialNo'] = serial_no
    
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

# 3️⃣ ทำความสะอาดข้อมูล
df['Measure Date Time'] = pd.to_datetime(df['Measure Date Time'])
numeric_cols = ['Battery Level', 'Temperature', 'Step', 'Calorie', 'Sleep Hour', 'Sleep Minute']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
df['Total Sleep (min)'] = df['Sleep Hour']*60 + df['Sleep Minute']
df['Date'] = df['Measure Date Time'].dt.date

# 4️⃣ แยกเวลานอนตาม Sleep State
sleep_states = df['Sleep State'].unique()
sleep_pivot = df.pivot_table(index=['SerialNo','Date'], 
                             columns='Sleep State', 
                             values='Total Sleep (min)', 
                             aggfunc='sum',
                             fill_value=0).reset_index()

# รวม Step, Calorie, Temperature, Battery
daily_activity = df.groupby(['SerialNo','Date']).agg({
    'Step': 'sum',
    'Calorie': 'sum',
    'Temperature': 'mean',
    'Battery Level': 'mean'
}).reset_index()

# รวมตาราง Sleep State
daily_activity = daily_activity.merge(sleep_pivot, on=['SerialNo','Date'], how='left')

# แปลงเวลานอนทั้งหมดเป็นชั่วโมง
for state in sleep_states:
    daily_activity[state+'_hr'] = daily_activity[state]/60

print(f"📊 Data loaded successfully!")
print(f"📈 Total records: {len(df):,}")
print(f"🔢 Devices: {df['SerialNo'].nunique()}")
print(f"📅 Date range: {df['Date'].min()} to {df['Date'].max()}")

In [ ]:
# 5️⃣ Seaborn-style Visualizations using Plotly

# 📊 1. Daily Steps per Device (Seaborn lineplot style)
fig = px.line(daily_activity, x='Date', y='Step', color='SerialNo',
              title='Daily Steps per Device',
              labels={'Step': 'Steps', 'Date': 'Date'},
              color_discrete_sequence=seaborn_colors,
              markers=True)

fig.update_layout(
    template='plotly_white',  # Clean seaborn-like background
    title_font_size=16,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    legend_title_text='Device',
    hovermode='x unified'
)

fig.update_traces(line=dict(width=2), marker=dict(size=6))
fig.show()

# 📊 2. Daily Calories per Device (Seaborn style)
fig2 = px.line(daily_activity, x='Date', y='Calorie', color='SerialNo',
               title='Daily Calories per Device',
               labels={'Calorie': 'Calories', 'Date': 'Date'},
               color_discrete_sequence=seaborn_colors,
               markers=True)

fig2.update_layout(
    template='plotly_white',
    title_font_size=16,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    legend_title_text='Device',
    hovermode='x unified'
)

fig2.update_traces(line=dict(width=2), marker=dict(size=6))
fig2.show()

In [ ]:
# 📊 3. Temperature Distribution by Device (Seaborn boxplot style)
fig3 = px.box(daily_activity, x='SerialNo', y='Temperature', 
              title='Temperature Distribution by Device',
              labels={'Temperature': 'Temperature (°C)', 'SerialNo': 'Device'},
              color='SerialNo',
              color_discrete_sequence=seaborn_colors)

fig3.update_layout(
    template='plotly_white',
    title_font_size=16,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    showlegend=False
)
fig3.show()

# 📊 4. Battery Level Over Time (Seaborn style)
fig4 = px.line(daily_activity, x='Date', y='Battery Level', color='SerialNo',
               title='Battery Level Over Time',
               labels={'Battery Level': 'Battery Level (%)', 'Date': 'Date'},
               color_discrete_sequence=seaborn_colors,
               markers=True)

fig4.update_layout(
    template='plotly_white',
    title_font_size=16,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14,
    legend_title_text='Device',
    hovermode='x unified'
)

fig4.update_traces(line=dict(width=2), marker=dict(size=6))
fig4.show()

In [ ]:
# 📊 5. Sleep State Stacked Bar Charts (Seaborn style) for each Device
sleep_hour_cols = [col for col in daily_activity.columns if '_hr' in col and col in daily_activity.columns]

if sleep_hour_cols:
    for device in daily_activity['SerialNo'].unique():
        device_data = daily_activity[daily_activity['SerialNo']==device].copy()
        
        if len(device_data) > 0:
            # Create stacked bar chart
            fig = go.Figure()
            
            # Add each sleep state as a bar
            colors = seaborn_colors[:len(sleep_hour_cols)]
            for i, col in enumerate(sleep_hour_cols):
                fig.add_trace(go.Bar(
                    name=col.replace('_hr', ''),
                    x=device_data['Date'],
                    y=device_data[col],
                    marker_color=colors[i % len(colors)]
                ))
            
            fig.update_layout(
                barmode='stack',
                title=f'Sleep State per Day - Device {device}',
                template='plotly_white',
                title_font_size=16,
                xaxis_title='Date',
                yaxis_title='Sleep Hours',
                xaxis_title_font_size=14,
                yaxis_title_font_size=14,
                legend_title_text='Sleep State'
            )
            
            fig.show()
else:
    print("No sleep hour columns found in the data")

In [ ]:
# 📊 6. Correlation Heatmap (Seaborn style)
numeric_columns = ['Step', 'Calorie', 'Temperature', 'Battery Level']
correlation_data = daily_activity[numeric_columns].corr()

# Create heatmap using plotly
fig_heatmap = go.Figure(data=go.Heatmap(
    z=correlation_data.values,
    x=correlation_data.columns,
    y=correlation_data.columns,
    colorscale='RdBu_r',  # Seaborn-like color scheme
    zmid=0,
    text=correlation_data.round(2).values,
    texttemplate="%{text}",
    textfont={"size": 12},
    hoverongaps=False
))

fig_heatmap.update_layout(
    title='Correlation Matrix of Activity Metrics',
    template='plotly_white',
    title_font_size=16,
    width=600,
    height=500,
    xaxis_title_font_size=14,
    yaxis_title_font_size=14
)

fig_heatmap.show()

# 📊 7. Multi-panel Dashboard (Seaborn FacetGrid style)
fig_dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Daily Steps', 'Daily Calories', 'Temperature', 'Battery Level'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]]
)

# Add traces for each metric
for i, device in enumerate(daily_activity['SerialNo'].unique()):
    device_data = daily_activity[daily_activity['SerialNo']==device]
    color = seaborn_colors[i % len(seaborn_colors)]
    
    # Steps
    fig_dashboard.add_trace(
        go.Scatter(x=device_data['Date'], y=device_data['Step'], 
                  mode='lines+markers', name=f'{device}', 
                  line=dict(color=color), showlegend=True),
        row=1, col=1
    )
    
    # Calories
    fig_dashboard.add_trace(
        go.Scatter(x=device_data['Date'], y=device_data['Calorie'], 
                  mode='lines+markers', name=f'{device}', 
                  line=dict(color=color), showlegend=False),
        row=1, col=2
    )
    
    # Temperature
    fig_dashboard.add_trace(
        go.Scatter(x=device_data['Date'], y=device_data['Temperature'], 
                  mode='lines+markers', name=f'{device}', 
                  line=dict(color=color), showlegend=False),
        row=2, col=1
    )
    
    # Battery
    fig_dashboard.add_trace(
        go.Scatter(x=device_data['Date'], y=device_data['Battery Level'], 
                  mode='lines+markers', name=f'{device}', 
                  line=dict(color=color), showlegend=False),
        row=2, col=2
    )

fig_dashboard.update_layout(
    title='Activity Metrics Dashboard',
    template='plotly_white',
    title_font_size=16,
    height=800,
    showlegend=True
)

fig_dashboard.show()